In [0]:
# Connect to ADLS
storage_account_name = "azureetlstorage"

storage_account_key = dbutils.secrets.get(
    scope = "etl-scope",
    key   = "adls-key"
)

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

print("connected!")

In [0]:
# Read Bronze orders
bronze_path = f"abfss://datalake@{storage_account_name}.dfs.core.windows.net/bronze/orders"

bronze_df = spark.read.format("delta").load(bronze_path)

print(f"Bronze rows: {bronze_df.count()}")
bronze_df.printSchema()

In [0]:
from pyspark.sql import functions as F

# Step 1 — Fix data types
typed_df = bronze_df \
    .withColumn("quantity",   F.col("quantity").cast("integer")) \
    .withColumn("unit_price", F.col("unit_price").cast("decimal(12,2)")) \
    .withColumn("order_date", F.to_date(F.col("order_date"), "yyyy-MM-dd")) \
    .withColumn("status",     F.lower(F.trim(F.col("status"))))

typed_df.printSchema()

In [0]:
# Step 2 — Flag bad rows
validated_df = typed_df \
    .withColumn(
        "_dq_issues",
        F.array_except(
            F.array(
                F.when(F.col("order_id").isNull(),    F.lit("NULL_ORDER_ID")).otherwise(F.lit(None)),
                F.when(F.col("quantity").isNull(),    F.lit("INVALID_QUANTITY")).otherwise(F.lit(None)),
                F.when(F.col("unit_price").isNull(),  F.lit("INVALID_PRICE")).otherwise(F.lit(None)),
                F.when(F.col("unit_price") < 0,       F.lit("NEGATIVE_PRICE")).otherwise(F.lit(None)),
                F.when(F.col("order_date").isNull(),  F.lit("INVALID_DATE")).otherwise(F.lit(None)),
            ),
            F.array(F.lit(None).cast("string"))
        )
    ) \
    .withColumn("_is_valid", F.size(F.col("_dq_issues")) == 0)

# Show invalid rows
invalid_df = validated_df.where(F.col("_is_valid") == False)
print(f"Invalid rows: {invalid_df.count()}")
invalid_df.show(5)

In [0]:
# Step 3 — Separate clean and quarantine rows
clean_df = validated_df.where(F.col("_is_valid") == True)
quarantine_df = validated_df.where(F.col("_is_valid") == False)

print(f"Clean rows:      {clean_df.count()}")
print(f"Quarantine rows: {quarantine_df.count()}")

# Save quarantine rows to ADLS for investigation
quarantine_path = f"abfss://datalake@{storage_account_name}.dfs.core.windows.net/bronze/orders_quarantine"

quarantine_df.write.format("delta") \
    .mode("overwrite") \
    .save(quarantine_path)

print("Quarantine rows saved!")

In [0]:
# Step 4 — Add derived column and deduplicate
from pyspark.sql import Window

# Add line total
clean_df = clean_df \
    .withColumn("line_total", F.col("quantity") * F.col("unit_price"))

# Deduplicate — keep latest row per order_id
window = Window.partitionBy("order_id").orderBy(F.desc("_ingested_at"))

deduped_df = clean_df \
    .withColumn("_row_num", F.row_number().over(window)) \
    .where(F.col("_row_num") == 1) \
    .drop("_row_num")

print(f"After dedup: {deduped_df.count()} rows")

In [0]:
# Step 5 — Write to Silver
silver_path = f"abfss://datalake@{storage_account_name}.dfs.core.windows.net/silver/orders"

silver_df = deduped_df \
    .withColumn("_silver_processed_at", F.current_timestamp())

silver_df.write.format("delta") \
    .mode("overwrite") \
    .save(silver_path)

print(f"Silver orders written — {silver_df.count()} rows")
silver_df.show(3)

**Read Customers from Bronze**

In [0]:
# Read Bronze customers
customers_bronze_path = f"abfss://datalake@{storage_account_name}.dfs.core.windows.net/bronze/customers"

customers_bronze_df = spark.read.format("delta").load(customers_bronze_path)

print(f"Bronze customers: {customers_bronze_df.count()}")
customers_bronze_df.printSchema()

In [0]:
from pyspark.sql import functions as F
# Silver customers — fix types and clean
silver_customers_df = customers_bronze_df \
    .withColumn("joined_date", F.to_date(F.col("joined_date"), "yyyy-MM-dd")) \
    .withColumn("tier", F.lower(F.trim(F.col("tier")))) \
    .withColumn("email", F.lower(F.trim(F.col("email")))) \
    .withColumn("_silver_processed_at", F.current_timestamp())

# Data quality checks
silver_customers_df = silver_customers_df \
    .withColumn(
        "_dq_issues",
        F.array_except(
            F.array(
                F.when(F.col("customer_id").isNull(), F.lit("NULL_CUSTOMER_ID")).otherwise(F.lit(None)),
                F.when(F.col("name").isNull(),        F.lit("NULL_NAME")).otherwise(F.lit(None)),
                F.when(F.col("email").isNull(),       F.lit("NULL_EMAIL")).otherwise(F.lit(None)),
                F.when(F.col("joined_date").isNull(), F.lit("INVALID_DATE")).otherwise(F.lit(None)),
                F.when(~F.col("tier").isin("bronze", "silver", "gold"), F.lit("INVALID_TIER")).otherwise(F.lit(None)),
            ),
            F.array(F.lit(None).cast("string"))
        )
    ) \
    .withColumn("_is_valid", F.size(F.col("_dq_issues")) == 0)

# Show results
invalid_customers = silver_customers_df.where(F.col("_is_valid") == False)
print(f"Invalid customers: {invalid_customers.count()}")
print(f"Valid customers:   {silver_customers_df.where(F.col('_is_valid') == True).count()}")

In [0]:
from pyspark.sql import Window
# Deduplicate
window = Window.partitionBy("customer_id").orderBy(F.desc("_ingested_at"))

clean_customers_df = silver_customers_df \
    .where(F.col("_is_valid") == True) \
    .withColumn("_row_num", F.row_number().over(window)) \
    .where(F.col("_row_num") == 1) \
    .drop("_row_num")

# Write to Silver
customers_silver_path = f"abfss://datalake@{storage_account_name}.dfs.core.windows.net/silver/customers"

clean_customers_df.write.format("delta") \
    .mode("overwrite") \
    .save(customers_silver_path)

print(f"Silver customers written — {clean_customers_df.count()} rows")

**Read Products from Bronze**

In [0]:
# Read Bronze products
products_bronze_path = f"abfss://datalake@{storage_account_name}.dfs.core.windows.net/bronze/products"

products_bronze_df = spark.read.format("delta").load(products_bronze_path)

print(f"Bronze products: {products_bronze_df.count()}")
products_bronze_df.printSchema()

In [0]:
# Silver products — validate and clean
silver_products_df = products_bronze_df \
    .withColumn("name", F.trim(F.col("name"))) \
    .withColumn("category", F.trim(F.col("category"))) \
    .withColumn("_silver_processed_at", F.current_timestamp())

# Valid categories
valid_categories = ["Electronics", "Clothing", "Food", "Books", "Sports", "Home", "Beauty"]

# Data quality checks
silver_products_df = silver_products_df \
    .withColumn(
        "_dq_issues",
        F.array_except(
            F.array(
                F.when(F.col("product_id").isNull(), F.lit("NULL_PRODUCT_ID")).otherwise(F.lit(None)),
                F.when(F.col("name").isNull(),       F.lit("NULL_NAME")).otherwise(F.lit(None)),
                F.when(F.col("unit_cost") < 0,       F.lit("NEGATIVE_COST")).otherwise(F.lit(None)),
                F.when(~F.col("category").isin(valid_categories), F.lit("INVALID_CATEGORY")).otherwise(F.lit(None)),
            ),
            F.array(F.lit(None).cast("string"))
        )
    ) \
    .withColumn("_is_valid", F.size(F.col("_dq_issues")) == 0)

invalid_products = silver_products_df.where(F.col("_is_valid") == False)
print(f"Invalid products: {invalid_products.count()}")
print(f"Valid products:   {silver_products_df.where(F.col('_is_valid') == True).count()}")

In [0]:
# Deduplicate
window = Window.partitionBy("product_id").orderBy(F.desc("_ingested_at"))

clean_products_df = silver_products_df \
    .where(F.col("_is_valid") == True) \
    .withColumn("_row_num", F.row_number().over(window)) \
    .where(F.col("_row_num") == 1) \
    .drop("_row_num")

# Write to Silver
products_silver_path = f"abfss://datalake@{storage_account_name}.dfs.core.windows.net/silver/products"

clean_products_df.write.format("delta") \
    .mode("overwrite") \
    .save(products_silver_path)

print(f"Silver products written — {clean_products_df.count()} rows")

In [0]:
# Read Silver customers and products
customers_silver_path = f"abfss://datalake@{storage_account_name}.dfs.core.windows.net/silver/customers"
products_silver_path  = f"abfss://datalake@{storage_account_name}.dfs.core.windows.net/silver/products"

silver_customers = spark.read.format("delta").load(customers_silver_path)
silver_products  = spark.read.format("delta").load(products_silver_path)

print("Customers and products loaded!")

In [0]:
# Read Silver orders from ADLS
orders_silver_path = f"abfss://datalake@{storage_account_name}.dfs.core.windows.net/silver/orders"

silver_orders = spark.read.format("delta").load(orders_silver_path)

# Join orders + customers + products
enriched_df = silver_orders \
    .join(silver_customers, on="customer_id", how="left") \
    .join(silver_products,  on="product_id",  how="left")

print(f"Enriched rows: {enriched_df.count()}")
enriched_df.printSchema()

In [0]:
# Select only relevant columns
enriched_df = enriched_df.select(
    # Order columns
    "order_id",
    "order_date",
    "status",
    "quantity",
    "unit_price",
    "line_total",

    # Customer columns
    "customer_id",
    silver_customers["name"].alias("customer_name"),
    "email",
    "country",
    "city",
    "tier",

    # Product columns
    "product_id",
    silver_products["name"].alias("product_name"),
    "category",
    "unit_cost",
    "is_active",

    # Metadata
    F.current_timestamp().alias("_enriched_at")
)

print(f"Final enriched rows: {enriched_df.count()}")
enriched_df.printSchema()

In [0]:
# Write enriched table to Silver
enriched_silver_path = f"abfss://datalake@{storage_account_name}.dfs.core.windows.net/silver/orders_enriched"

enriched_df.write.format("delta") \
    .mode("overwrite") \
    .save(enriched_silver_path)

print(f"Silver enriched table written — {enriched_df.count()} rows")